In [120]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt


In [168]:
ROOT_DIR = os.path.dirname(os.getcwd()) 
json_path = os.path.join(ROOT_DIR, "results/optuna_results/mpn_mmb_desc/dili/log/feature_importance_mean.json")

output_dir = os.path.join(ROOT_DIR, "results/optuna_results/mpn_mmb_desc/dili/log/feature_importance")
os.makedirs(output_dir, exist_ok=True)

with open(json_path, "r") as f:
    data = json.load(f)

dataset_name = os.path.basename(os.path.dirname(os.path.dirname(json_path)))
metric = data.get("metric", "Unknown")
direction = data.get("metric_direction", "")
direction_symbol = "↓" if direction == "min" else "↑" if direction == "max" else ""

# check the content
for k, v in data.items():
    print(f"{k}: {len(v)} dimensions")


mmb_deltas: 256 dimensions
mpn_deltas: 64 dimensions
desc_deltas: 200 dimensions
metric: 7 dimensions
metric_direction: 3 dimensions


### The importance of each dimension for a single modality (bar chart)

In [169]:
def plot_feature_bars(deltas, component):
    plt.figure(figsize=(12, 4), dpi=300)
    x = np.arange(len(deltas))
    plt.bar(x, deltas, color='skyblue')
    
    plt.xlabel("Embedding Dimension Index")
    plt.ylabel("% Performance Drop")
    
    title = (f"{dataset_name} - {component.upper()} Feature Importance (Avg across Seeds)\n"
             f"(Metric: {metric} {direction_symbol})")
    plt.title(title)
    
    plt.grid(axis='y')
    plt.tight_layout()
    
    save_path = os.path.join(output_dir, f"{component}_feature_importance.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()



### The average contribution of the three modalities (bar chart)

In [170]:
def plot_average_bar(avg_result):
    modality_keys = ["mmb_deltas", "mpn_deltas", "desc_deltas"]
    
    name_map = {
        "mmb": "MMB",
        "mpn": "D-MPNN",
        "desc": "DESC"
    }
    
    avg_values = {}
    for key in modality_keys:
        if key in avg_result:
            try:
                values = [float(v) for v in avg_result[key]]
                comp_name = key.split("_")[0]
                # avg_values[name_map.get(comp_name, comp_name.upper())] = np.mean(values)
                avg_values[name_map.get(comp_name, comp_name.upper())] = np.mean(np.abs(values))
            except ValueError:
                print(f"[Warning] Cannot convert: {key}")
  
    # Keep a fixed plotting order
    ordered_keys = ["mpn", "mmb", "desc"]
    modalities = [name_map[k] for k in ordered_keys if name_map[k] in avg_values]
    means = [avg_values[name_map[k]] for k in ordered_keys if name_map[k] in avg_values]

    plt.figure(figsize=(6, 3), dpi=300)
    bars = plt.bar(modalities, means, edgecolor='black')

    # plt.ylabel("Mean % Performance Drop", fontsize=12)
    # plt.ylabel("Mean Absolute % Performance Impact", fontsize=12)
    plt.ylabel("Performance Impact (%)", fontsize=12)
    plt.xlabel("Modality", fontsize=12)
    plt.title("Mean Feature Importance by Modality", fontsize=14)

    plt.tight_layout()
    
    save_path = os.path.join(output_dir, "average_feature_importance.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"[Saved] {save_path}")

    plt.close()


### Plot

In [171]:
for comp in ["mmb_deltas", "mpn_deltas", "desc_deltas"]:
    if comp in data:
        comp_name = comp.split("_")[0]
        plot_feature_bars(data[comp], component=name_map.get(comp_name, comp_name))

plot_average_bar(data)

[Saved] /workspace/00_ADMET_prediction_v6/results/optuna_results/mpn_mmb_desc/dili/log/feature_importance/average_feature_importance.png
